# Agent with memory and reflextion pattern

Before start, configure an account in this platforms and get apikeys:
- LangSmith (LANGSMITH_API_KEY)
- LangFuse (LANGFUSE_PUBLICEY, LANGFUSE_SECRETKEY)
- OpenAI (OPENAI_API_KEY)
- SerpAPI (SERPAPI_API_KEY)

*¿Qué es Reflexion Agent?**

Un agente que no solo ejecuta tareas, sino que aprende de sus errores. Combina:

- Memoria Dual: Corto plazo (conversacional) y largo plazo (vectorial)
- Auto-reflexión: Analiza éxitos y fallos para mejorar
- Recuperación Semántica: Usa experiencias previas como contexto

In [ ]:
! pip install -U langchain langchain_community langchain_mcp_adapters langfuse langchain-openai langchain-google-genai dotenv google-search-results arxiv faiss-cpu

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import display, Markdown
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import Tool, tool
from langchain_community.utilities import SerpAPIWrapper
from langchain_community.utilities import ArxivAPIWrapper
from langchain_mcp_adapters.client import MultiServerMCPClient

from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
import os

from datetime import datetime
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Logging with Langfuse
Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLICEY"),
    secret_key=os.getenv("LANGFUSE_SECRETKEY"),
    host="https://us.cloud.langfuse.com"
)
langfuse = get_client()
langfuse_handler = CallbackHandler()

/tmp/ipykernel_566918/691122593.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SerpAPIWrapper


#### Ejemplo sin memoria

In [2]:
# Para GPT (OpenAI) # OPENAI_API_KEY must be a envar
llm = init_chat_model(
    model="gpt-4.1-nano", 
    model_provider="openai",
    temperature=0.7,
    timeout=30,
    max_tokens=1000
    )

# Para Gemini (Google) # conseguir la GEMINI_API_KEY de Gemini Studio
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

response = llm.invoke("Hola, me llamo Vicente")
#print(response.content)
display(Markdown(response.content))

¡Hola, Vicente! Mucho gusto.

¿En qué puedo ayudarte hoy?

In [3]:
response = llm.invoke("¿Como me llamo?")
display(Markdown(response.content))

Como soy una inteligencia artificial, no tengo acceso a tu información personal ni sé cuál es tu nombre. No tengo forma de saber cómo te llamas.

Si quieres, puedes decírmelo, ¡pero no lo recordaré para futuras interacciones, ya que no guardo datos personales!

#### Simple example short term memory

Saving conversations

In [14]:
# Create in-memory chat history (SHORT-TERM)
chat_history = InMemoryChatMessageHistory()

# Add messages
chat_history.add_user_message("Hi, my name is Alice")
chat_history.add_ai_message("Hello Alice! Nice to meet you!")

# Retrieve messages
for msg in chat_history.messages:
    print(f"[{msg.type.upper()}]: {msg.content}")

# Output:
# [HUMAN]: Hi, my name is Alice
# [AI]: Hello Alice! Nice to meet you!

[HUMAN]: Hi, my name is Alice
[AI]: Hello Alice! Nice to meet you!


Example with agent

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Be concise and friendly."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

llm = init_chat_model(
    model="gpt-4.1-nano", 
    model_provider="openai",
    temperature=0.7,
    timeout=30,
    max_tokens=1000
    )

chain = prompt | llm

agent_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

/home/vicente/anaconda3/envs/agents/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [18]:
config = {"configurable": {"session_id": "user_125"}}

response1 = agent_with_history.invoke(    {"input": "Hi! My name is pepe and I love Python programming."},    config=config)
print(f"User: Hi! My name is pepe and I love Python programming.")
print(f"AI: {response1.content}\n")

response2 = agent_with_history.invoke(    {"input": "What's my name and what do I love?"},    config=config)
print(f"User: What's my name and what do I love?")
print(f"AI: {response2.content}\n")

print("--- Stored Chat History ---")
history = get_session_history("user_123")
for msg in history.messages:
    print(f"{msg.type}: {msg.content}")

User: Hi! My name is pepe and I love Python programming.
AI: Hi Pepe! That's great to hear—Python is a fantastic language. How can I help you today?

User: What's my name and what do I love?
AI: Your name is Pepe, and you love Python programming.

--- Stored Chat History ---
human: Hi! My name is Alice and I love Python programming.
ai: Hi Alice! That's great to hear. Python is a fantastic language. Do you need any help with Python programming?
human: What's my name and what do I love?
ai: Your name is Alice, and you love Python programming.
human: Hi! My name is Alice and I love Python programming.
ai: Hi Alice! It's great to meet a fellow Python enthusiast. 😊
human: What's my name and what do I love?
ai: Your name is Alice, and you love Python programming.
human: Hi! My name is Vicente and I love Python programming.
ai: Hi Vicente! It's awesome that you love Python programming. 😊
human: What's my name and what do I love?
ai: Your name is Vicente, and you love Python programming.
huma

#### Long term memory with FAISS

FAISS dataset initialization and util functions to manage memory

In [19]:
embeddings = OpenAIEmbeddings()
FAISS_PATH = "memory_faiss_index"

try:
    # Intentar cargar memoria existente
    vector_store = FAISS.load_local(
        FAISS_PATH, 
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("Memoria cargada desde disco")
except:
    # Crear nueva memoria vacía
    vector_store = FAISS.from_texts(
        ["Inicio de la memoria del asistente."],
        embeddings,
        metadatas=[{"type": "system", "timestamp": str(datetime.now())}]
    )
    print("Nueva memoria creada")

def save_to_memory(text: str, memory_type: str = "conversation"):
    doc = Document(
        page_content=text,
        metadata={
            "type": memory_type,
            "timestamp": str(datetime.now())
        }
    )
    vector_store.add_documents([doc])
    vector_store.save_local(FAISS_PATH)
    print(f"Guardado en memoria: {text[:50]}...")


def search_memory(query: str, k: int = 3) -> list:
    results = vector_store.similarity_search(query, k=k)
    return [doc.page_content for doc in results]


Nueva memoria creada


The agent

In [20]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente con memoria a largo plazo.
Usas los recuerdos relevantes para personalizar tus respuestas.

RECUERDOS RELEVANTES:
{memories}

Responde de forma amigable y usa la información de los recuerdos cuando sea relevante."""),
    ("human", "{input}")
])

llm = init_chat_model(
    model="gpt-4.1-nano", 
    model_provider="openai",
    temperature=0.7,
    timeout=30,
    max_tokens=1000
)

def chat(user_input: str) -> str:    
    # Buscar recuerdos relevantes
    memories = search_memory(user_input, k=3)
    memories_text = "\n".join([f"- {m}" for m in memories])
    
    # Crear el mensaje con contexto
    chain = prompt | llm
    response = chain.invoke({
        "memories": memories_text,
        "input": user_input
    })
    
    # Guardar la conversación en memoria
    save_to_memory(f"Usuario dijo: {user_input}")
    save_to_memory(f"Asistente respondió: {response.content}")
    
    return response.content

In [21]:
print("\n" + "="*50)
print("SESIÓN DE CHAT CON MEMORIA SEMÁNTICA")
print("="*50)

# Conversación 1
print("\nUsuario: Hola, me llamo Carlos y soy ingeniero de software.")
r1 = chat("Hola, me llamo Carlos y soy ingeniero de software.")
print(f"AI: {r1}")

# Conversación 2
print("\nUsuario: Mi comida favorita es la pizza con piña.")
r2 = chat("Mi comida favorita es la pizza con piña.")
print(f"AI: {r2}")

# Conversación 3
print("\nUsuario: Tengo un perro llamado Max.")
r3 = chat("Tengo un perro llamado Max.")
print(f"AI: {r3}")

print("\n" + "="*50)
print("[SIMULAR REINICIO DEL PROGRAMA]")
print("="*50)

# Preguntas que requieren memoria
print("\nUsuario: ¿Cómo se llama mi mascota?")
r4 = chat("¿Cómo se llama mi mascota?")
print(f"AI: {r4}")  # Debería recordar: Max

print("\nUsuario: ¿Qué me gusta comer?")
r5 = chat("¿Qué me gusta comer?")
print(f"AI: {r5}")  # Debería recordar: pizza con piña

print("\nUsuario: ¿A qué me dedico profesionalmente?")
r6 = chat("¿A qué me dedico profesionalmente?")
print(f"AI: {r6}")  # Debería recordar: ingeniero de software


SESIÓN DE CHAT CON MEMORIA SEMÁNTICA

Usuario: Hola, me llamo Carlos y soy ingeniero de software.
Guardado en memoria: Usuario dijo: Hola, me llamo Carlos y soy ingenier...
Guardado en memoria: Asistente respondió: ¡Hola, Carlos! Encantado de c...
AI: ¡Hola, Carlos! Encantado de conocerte. Como ingeniero de software, seguro tienes muchas ideas interesantes y proyectos en mente. ¿En qué puedo ayudarte hoy?

Usuario: Mi comida favorita es la pizza con piña.
Guardado en memoria: Usuario dijo: Mi comida favorita es la pizza con p...
Guardado en memoria: Asistente respondió: ¡Qué deliciosa elección, Carl...
AI: ¡Qué deliciosa elección, Carlos! La pizza con piña es una combinación que muchos disfrutan por su sabor dulce y salado. ¿Tienes alguna pizza favorita o algún otro platillo que también te guste mucho?

Usuario: Tengo un perro llamado Max.
Guardado en memoria: Usuario dijo: Tengo un perro llamado Max....
Guardado en memoria: Asistente respondió: ¡Qué bonito nombre para un pe...
AI: ¡Q

In [10]:
# Conversación 1
print("\nUsuario: Me especializo en desarrollo de backend en Python.")
r1 = chat("Me especializo en desarrollo de backend en Python.")
print(f"AI: {r1}")

# Conversación 2
print("\nUsuario: Me gustaría aprender Java")
r2 = chat("Me gustaría aprender Java")
print(f"AI: {r2}")


Usuario: Me especializo en desarrollo de backend en Python.
Guardado en memoria: Usuario dijo: Me especializo en desarrollo de back...
Guardado en memoria: Asistente respondió: ¡Qué genial, Carlos! Como esp...
AI: ¡Qué genial, Carlos! Como especialista en desarrollo de backend en Python, seguramente tienes muy buenas habilidades y conocimientos en ese área. ¿Hay algún proyecto en particular en el que estés trabajando o alguna duda en la que pueda ayudarte hoy?

Usuario: Me gustaría aprender Java
Guardado en memoria: Usuario dijo: Me gustaría aprender Java...
Guardado en memoria: Asistente respondió: ¡Qué buena decisión, Carlos! ...
AI: ¡Qué buena decisión, Carlos! Java es un lenguaje muy popular y versátil, ideal para desarrollar desde aplicaciones web hasta móviles con Android. Como ya tienes experiencia en desarrollo de backend en Python, seguramente te será más fácil entender conceptos como programación orientada a objetos y estructuras de datos en Java. ¿Te gustaría que te recomie

In [11]:
# Conversación 3
print("\nUsuario: Cuales mis lenguajes de programación favoritos?")
r2 = chat("Cuales mis lenguajes de programación favoritos?")
print(f"AI: {r2}")


Usuario: Cuales mis lenguajes de programación favoritos?
Guardado en memoria: Usuario dijo: Cuales mis lenguajes de programación...
Guardado en memoria: Asistente respondió: ¡Hola, Carlos! Según lo que r...
AI: ¡Hola, Carlos! Según lo que recuerdo, no has mencionado explícitamente tus lenguajes de programación favoritos, pero sí sé que tienes experiencia en desarrollo de backend en Python y que estás interesado en aprender Java. Si quieres, puedo ayudarte a explorar tus preferencias o recomendarte recursos para seguir creciendo en esos lenguajes. ¿Te gustaría contarme cuáles son tus favoritos o en qué te gustaría enfocarte más?


#### Full example with reflection

In [24]:
# ============================================================
# AGENTE CON MEMORIA REFLEXIVA
# Short-term + Long-term Memory con Reflexiones
# ============================================================

import os
import json
from datetime import datetime
from typing import List, Dict
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.documents import Document
from dotenv import load_dotenv
from IPython.display import display, Markdown
load_dotenv()

# ============================================================
# CONFIGURACIÓN
# ============================================================
FAISS_PATH = "reflections_memory"
REFLECTION_INTERVAL = 3  # Generar reflexión cada N mensajes
MAX_SHORT_TERM_MESSAGES = 10  # Máximo mensajes en memoria corta
TOP_K_REFLECTIONS = 5  # Número de reflexiones a recuperar

# ============================================================
# PROMPTS
# ============================================================

# Prompt para generar reflexiones
REFLECTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Eres un analizador de conversaciones. Tu tarea es extraer 
información importante sobre el usuario a partir de la conversación.

Analiza la conversación y genera reflexiones estructuradas sobre el usuario.
Cada reflexión debe ser una oración clara y concisa.

CATEGORÍAS A IDENTIFICAR:
- IDENTIDAD: Nombre, edad, ubicación, profesión
- PREFERENCIAS: Gustos, favoritos, intereses
- PERSONALIDAD: Rasgos de carácter, estilo de comunicación
- RELACIONES: Familia, amigos, mascotas
- OBJETIVOS: Metas, proyectos, planes
- HECHOS: Información factual mencionada

Formato de salida (JSON):
{{
    "reflections": [
        {{
            "category": "CATEGORIA",
            "content": "Reflexión sobre el usuario",
            "confidence": "alta/media/baja"
        }}
    ]
}}

Si no hay información nueva relevante, devuelve: {{"reflections": []}}
"""),
    ("human", """Analiza esta conversación y extrae reflexiones sobre el usuario:

{conversation}

Genera las reflexiones en formato JSON:""")
])

# Prompt principal del agente
AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente personal inteligente con memoria a largo plazo.
Tienes acceso a dos tipos de memoria:

1. REFLEXIONES (Memoria a largo plazo - conocimiento sobre el usuario):
{reflections}

2. CONVERSACIÓN RECIENTE (Contexto inmediato):
Está en el historial de mensajes.

INSTRUCCIONES:
- Usa las reflexiones para personalizar tus respuestas
- Recuerda detalles sobre el usuario naturalmente
- Sé amigable y demuestra que recuerdas información previa
- No menciones explícitamente que "tienes memoria" o "recuerdas"
- Integra el conocimiento de forma natural en la conversación
"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])


# ============================================================
# CLASE: ReflectiveMemoryAgent
# ============================================================
class ReflectiveMemoryAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7)
        self.reflection_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.embeddings = OpenAIEmbeddings()
        
        # Memoria a corto plazo
        self.short_term_memory = InMemoryChatMessageHistory()
        self.message_count = 0
        
        # Memoria a largo plazo (FAISS)
        self.vector_store = self._load_or_create_vector_store()
        
        # Chains
        self.agent_chain = AGENT_PROMPT | self.llm
        self.reflection_chain = REFLECTION_PROMPT | self.reflection_llm
    
    def _load_or_create_vector_store(self) -> FAISS:
        try:
            vs = FAISS.load_local(
                FAISS_PATH,
                self.embeddings,
                allow_dangerous_deserialization=True
            )
            print("Memoria de reflexiones cargada")
            return vs
        except:
            vs = FAISS.from_texts(
                ["Sistema de memoria inicializado."],
                self.embeddings,
                metadatas=[{"category": "SYSTEM", "timestamp": str(datetime.now())}]
            )
            vs.save_local(FAISS_PATH)
            print("Nueva memoria de reflexiones creada")
            return vs
    
    def _get_relevant_reflections(self, query: str) -> str:
        results = self.vector_store.similarity_search(query, k=TOP_K_REFLECTIONS)
        
        if not results:
            return "No hay reflexiones previas sobre el usuario."
        
        reflections = []
        for doc in results:
            if doc.metadata.get("category") != "SYSTEM":
                category = doc.metadata.get("category", "GENERAL")
                reflections.append(f"[{category}] {doc.page_content}")
        
        if not reflections:
            return "No hay reflexiones relevantes."
        
        return "\n".join(reflections)
    
    def _generate_reflections(self) -> List[Dict]:
        # Formatear conversación reciente
        messages = self.short_term_memory.messages[-6:]  # Últimos 6 mensajes
        conversation = ""
        for msg in messages:
            role = "Usuario" if msg.type == "human" else "Asistente"
            conversation += f"{role}: {msg.content}\n"
        
        if not conversation.strip():
            return []
        
        # Generar reflexiones
        try:
            response = self.reflection_chain.invoke({"conversation": conversation})
            
            # Parsear JSON
            content = response.content
            # Limpiar posibles marcadores de código
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]
            
            data = json.loads(content.strip())
            return data.get("reflections", [])
        except Exception as e:
            print(f"Error generando reflexiones: {e}")
            return []
    
    def _save_reflections(self, reflections: List[Dict]):
        if not reflections:
            return
        
        documents = []
        for ref in reflections:
            doc = Document(
                page_content=ref["content"],
                metadata={
                    "category": ref.get("category", "GENERAL"),
                    "confidence": ref.get("confidence", "media"),
                    "timestamp": str(datetime.now())
                }
            )
            documents.append(doc)
            print(f"Reflexión guardada [{ref.get('category')}]: {ref['content'][:50]}...")
        
        self.vector_store.add_documents(documents)
        self.vector_store.save_local(FAISS_PATH)
    
    def _trim_short_term_memory(self):
        messages = self.short_term_memory.messages
        if len(messages) > MAX_SHORT_TERM_MESSAGES:
            # Mantener solo los últimos N mensajes
            self.short_term_memory.clear()
            for msg in messages[-MAX_SHORT_TERM_MESSAGES:]:
                if msg.type == "human":
                    self.short_term_memory.add_user_message(msg.content)
                else:
                    self.short_term_memory.add_ai_message(msg.content)
    
    def chat(self, user_input: str) -> str:
        self.message_count += 1
        
        # 1. Recuperar reflexiones relevantes
        reflections = self._get_relevant_reflections(user_input)
        
        # 2. Obtener historial de chat (memoria corto plazo)
        chat_history = self.short_term_memory.messages
        
        # 3. Generar respuesta
        response = self.agent_chain.invoke({
            "reflections": reflections,
            "chat_history": chat_history,
            "input": user_input
        })
        
        # 4. Guardar en memoria a corto plazo
        self.short_term_memory.add_user_message(user_input)
        self.short_term_memory.add_ai_message(response.content)
        
        # 5. Generar reflexiones periódicamente
        if self.message_count % REFLECTION_INTERVAL == 0:
            print("\nGenerando reflexiones...")
            new_reflections = self._generate_reflections()
            self._save_reflections(new_reflections)
        
        # 6. Limpiar memoria corta si es necesario
        self._trim_short_term_memory()
        
        return response.content
    
    def get_all_reflections(self) -> List[str]:
        """Obtiene todas las reflexiones almacenadas."""
        results = self.vector_store.similarity_search("usuario información", k=100)
        return [
            f"[{doc.metadata.get('category', 'GENERAL')}] {doc.page_content}"
            for doc in results
            if doc.metadata.get("category") != "SYSTEM"
        ]
    
    def force_reflection(self):
        """Fuerza la generación de reflexiones."""
        print("\nForzando generación de reflexiones...")
        reflections = self._generate_reflections()
        self._save_reflections(reflections)
        return reflections

In [25]:
agent = ReflectiveMemoryAgent()
    
print("\n" + "="*60)
print("AGENTE CON MEMORIA REFLEXIVA")
print("="*60)

# Sesión de conversación
conversations = [
    "Hola! Me llamo María y soy diseñadora gráfica.",
    "Vivo en Barcelona con mi gato Michi.",
    "Me encanta el café con leche de avena por las mañanas.",
    "Estoy aprendiendo a programar en Python.",
    "Mi color favorito es el azul turquesa.",
    "Los fines de semana me gusta ir a la playa.",
]

for msg in conversations:
    print(f"\nUsuario: {msg}")
    response = agent.chat(msg)
    print(f"Asistente: {response}")

# Forzar reflexión final
agent.force_reflection()

print("\n" + "="*60)
print("REFLEXIONES ALMACENADAS")
print("="*60)
for ref in agent.get_all_reflections():
    print(f"  • {ref}")

print("\n" + "="*60)
print("SIMULANDO REINICIO DEL PROGRAMA...")
print("="*60)

# Crear nuevo agente (simula reinicio)
agent2 = ReflectiveMemoryAgent()

# Preguntas que requieren memoria a largo plazo
test_questions = [
    "¿Recuerdas cómo me llamo?",
    "¿Qué mascota tengo?",
    "¿Qué estoy aprendiendo?",
    "¿Cuál es mi profesión?",
]

for q in test_questions:
    print(f"\nUsuario: {q}")
    response = agent2.chat(q)
    print(f"Asistente: {response}")

Memoria de reflexiones cargada

AGENTE CON MEMORIA REFLEXIVA

Usuario: Hola! Me llamo María y soy diseñadora gráfica.
Asistente: ¡Hola, María! Qué gusto saludarte. Ser diseñadora gráfica debe ser una profesión muy creativa e interesante. ¿En qué tipo de proyectos estás trabajando últimamente?

Usuario: Vivo en Barcelona con mi gato Michi.
Asistente: ¡Qué bien, María! Barcelona es una ciudad increíble, y seguro que Michi disfruta mucho de la vida contigo allí. ¿Es travieso o más bien tranquilo? Me imagino que debe ser una compañía genial mientras trabajas en tus diseños.

Usuario: Me encanta el café con leche de avena por las mañanas.

Generando reflexiones...
Reflexión guardada [IDENTIDAD]: El nombre del usuario es María, es diseñadora gráf...
Reflexión guardada [PREFERENCIAS]: María disfruta del café con leche de avena por las...
Reflexión guardada [RELACIONES]: María tiene un gato llamado Michi que vive con ell...
Reflexión guardada [PERSONALIDAD]: María parece ser una persona creati